# Step 1.1 — Data Sourcing

**Goal:** Load all **currently active cantonal laws** from the *Systematische Gesetzessammlung Basel-Stadt (SGBS)* via the REST API on [data.bs.ch](https://data.bs.ch) (Dataset ID: `100354`) and persist them locally as a CSV file.

**Why this approach?**  
The dataset is publicly available and machine-readable via a REST API — no manual download or labelling required. We apply two filters:
1. **Active laws only** (`is_active=True`): repealed laws are irrelevant to our research question about the *current* semantic structure of Basel-Stadt legislation.
2. **Cantonal laws only** (exclude `category_id` starting with `9`): category `9.x` covers municipal (*Gemeinde*) legislation, which follows a different legislative logic and would introduce noise into our clustering.

Instead of paginating through records (hard API limit of 10'000), we use the **export endpoint**, which returns the full dataset as a single CSV. We persist the raw data to `data/raw/` so subsequent steps work offline and reproducibly.

## 1. Setup

In [1]:
import requests
import pandas as pd
import io
import os

RAW_DATA_PATH = "../data/raw/sgbs_raw.csv"
os.makedirs("../data/raw", exist_ok=True)

## 2. API Configuration

In [2]:
EXPORT_URL = "https://data.bs.ch/api/explore/v2.1/catalog/datasets/100354/exports/csv"

params = {
    "delimiter": ",",
    "use_labels": False
}

## 3. Fetch and Load Data

In [3]:
response = requests.get(EXPORT_URL, params=params)
response.raise_for_status()

df = pd.read_csv(io.StringIO(response.text))
print(f"Total records loaded: {len(df)}")

Total records loaded: 10944


## 4. Filter: Active Laws Only

The `is_active` filter has no effect on the export endpoint, so we apply it in pandas.

In [5]:
before = len(df)
df = df[df["is_active"] == True]
print(f"After is_active filter: {len(df)} (removed {before - len(df)})")

After is_active filter: 908 (removed 0)


## 5. Filter: Cantonal Laws Only

`category_id` values starting with `9` belong to municipal (*Gemeinde*) legislation (e.g. Riehen, Bettingen). We exclude these to focus exclusively on cantonal law, which has a consistent legislative structure suitable for our clustering analysis.

In [6]:
before = len(df)
df = df[~df["category_id"].astype(str).str.startswith("9")]
print(f"After cantonal filter: {len(df)} (removed {before - len(df)} municipal records)")

After cantonal filter: 739 (removed 169 municipal records)


## 6. Select Relevant Columns

| Column | Purpose |
|--------|--------|
| `title_de` | Full law title — human-readable label |
| `keywords_de` | Short title / keywords — used for visualizations and tables |
| `text_of_law` | Full legal text — primary NLP input |
| `systematic_number` | Official SGBS category number — used for **validation** (human logic vs. ML logic) |
| `category_name` | Human-readable category name |
| `category_id` | Numeric category ID — retained for potential further filtering |

In [8]:
COLS_TO_KEEP = ["title_de", "keywords_de", "text_of_law", "systematic_number", "category_name", "category_id"]

available_cols = [c for c in COLS_TO_KEEP if c in df.columns]
print(f"Keeping columns: {available_cols}")

df_slim = df[available_cols].copy()

before = len(df_slim)
df_slim = df_slim[df_slim["text_of_law"].notna() & (df_slim["text_of_law"].str.strip() != "")]
print(f"After dropping empty text: {len(df_slim)} (removed {before - len(df_slim)})")

df_slim.head(3)

Keeping columns: ['title_de', 'keywords_de', 'text_of_law', 'systematic_number', 'category_name', 'category_id']
After dropping empty text: 733 (removed 6)


,title_de,keywords_de,text_of_law,systematic_number,category_name,category_id
3,Entfernung der Spitalliste aus der Gesetzessam...,NaN,Spitäler\n\n\n 330.500\n\n\n Entfernung der ...,330.500,Anderes,5.0
37,Staatsvertrag zwischen den Kantonen Basel-Stad...,NaN,Gesundheitsversorgung: Staatsvertrag BL + BS |...,333.200,Interkantonale Vereinbarung,2.0
38,Verordnung über die von der Sanität Basel zu e...,Gebührenverordnung Sanität,Sanität: Gebührenverordnung | Sanitätsdienst\n...,339.220,Verordnung,6.0


## 7. Persist Raw Data

In [9]:
df_slim.to_csv(RAW_DATA_PATH, index=False)
print(f"Saved {len(df_slim)} records to {RAW_DATA_PATH}")

Saved 733 records to ../data/raw/sgbs_raw.csv


## 8. Summary

**What we did:**
- Fetched all laws from the SGBS export endpoint
- Filtered for active laws (`is_active=True`)
- Filtered for cantonal laws only (excluded `category_id` starting with `9`)
- Dropped records with empty `text_of_law`
- Retained columns: `title_de`, `keywords_de`, `text_of_law`, `systematic_number`, `category_name`, `category_id`
- Persisted as `data/raw/sgbs_raw.csv`

**Next step:** `01_2_html_cleaning.ipynb` — remove HTML tags from `text_of_law`.